# CrewAI: Complete Guide to Multi-Agent AI Systems


---

## What is CrewAI?

**CrewAI** is the leading open-source Python framework for orchestrating autonomous AI agents and building complex, production-ready workflows. It enables teams of AI agents to collaborate on tasks that are too complex for a single agent.

| Concept | Purpose |
|---------|---------|
| **Agent** | Autonomous unit with a role, goal, backstory, and tools |
| **Task** | A specific assignment given to an agent |
| **Crew** | A team of agents executing a set of tasks together |
| **Process** | How tasks execute — sequential or hierarchical |


# 1: Setup and Installation

## Required Packages

| Package | Purpose |
|---------|---------|
| `crewai` | Core framework: Agents, Tasks, Crews |
| `crewai-tools` | Pre-built tools and MCP adapter |

## LLMs Used in This Notebook

All examples use **Gemini 2.5 Flash** (`gemini/gemini-2.5-flash`).

CrewAI supports any LLM via LiteLLM — the model string format is `provider/model-name`:

```
gemini/gemini-2.5-flash        # Google Gemini (used here)
gpt-4o-mini                    # OpenAI
anthropic/claude-3-5-haiku-20241022  # Anthropic
ollama/llama3.2                # Local via Ollama (free)
groq/llama-3.1-70b-versatile   # Groq (fast inference)
```

In [ ]:
# Install CrewAI and tools
! pip install crewai crewai-tools -q

In [ ]:
import os
from google.colab import userdata


# Set your Google API key
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

# Optional: Serper API key for web search tools
# os.environ["SERPER_API_KEY"] = "your-serper-api-key-here"

# Model used throughout this notebook
MODEL = "gemini/gemma-4-31b-it"

print("Environment configured.")
print(f"Model: {MODEL}")

# 2: Agents

## What is an Agent?

An **Agent** is an autonomous unit that can:
- Perform specific tasks
- Make decisions based on its role and goal
- Use tools to accomplish objectives
- Collaborate with other agents
- Maintain memory of interactions
- Delegate tasks when allowed

Think of an agent as a **specialized team member** — a researcher, writer, analyst, or editor — each with distinct expertise.

## Core Properties

```
AGENT
  role      ->  What the agent IS (job title / domain)
  goal      ->  What the agent is trying to achieve
  backstory ->  Why the agent is qualified (shapes reasoning)
  llm       ->  Which language model the agent uses
```

## Key Attributes

| Attribute | Type | Default | Description |
|-----------|------|---------|-------------|
| `role` | str | required | The agent's function and expertise |
| `goal` | str | required | The individual objective guiding decisions |
| `backstory` | str | required | Context and persona enriching interactions |
| `llm` | str/LLM | `gpt-4o` | Language model powering the agent |
| `tools` | List | `[]` | Capabilities available to the agent |
| `memory` | bool | False | Maintain conversation history |
| `verbose` | bool | False | Enable detailed execution logs |
| `max_iter` | int | 20 | Max iterations before giving best answer |
| `allow_delegation` | bool | False | Allow the agent to delegate tasks |
| `max_rpm` | int | None | Rate limit for API calls |

> **Tip:** The more specific the `role`, `goal`, and `backstory`, the better the agent performs. Write them like a detailed job description.

In [ ]:
from crewai import Agent

# Example 1: Basic agent
researcher = Agent(
    role="Senior Research Analyst",
    goal="Uncover cutting-edge developments in AI and summarize them clearly",
    backstory=(
        "You are a veteran analyst with 10+ years tracking the AI industry. "
        "Known for distilling complex technical papers into actionable insights, "
        "you always back claims with credible sources."
    ),
    llm="gemini/gemini-2.5-flash",
    verbose=True,
)

# Example 2: Agent with iteration limit
writer = Agent(
    role="Technical Content Writer",
    goal="Transform research findings into engaging, developer-friendly articles",
    backstory=(
        "With a background in both computer science and journalism, you excel at "
        "making complex AI concepts accessible. You write for technical audiences "
        "who value precision and practical examples."
    ),
    llm="gemini/gemini-2.5-flash",
    verbose=True,
    max_iter=15,
)

# Example 3: Agent with delegation enabled
manager = Agent(
    role="Editorial Director",
    goal="Oversee the content pipeline and ensure quality standards are met",
    backstory=(
        "20 years of editorial experience at top tech publications. "
        "You identify what makes content resonate with readers and ensure "
        "all output meets publication standards."
    ),
    llm="gemini/gemini-2.5-flash",
    allow_delegation=True,
    verbose=True,
)

print("Agents created.")
print(f"  researcher  : {researcher.role} | {researcher.llm}")
print(f"  writer      : {writer.role} | {writer.llm}")
print(f"  manager     : {manager.role} | allow_delegation={manager.allow_delegation}")

In [ ]:
# uncomment and run
# print(researcher)

# 3: Tasks

## What is a Task?

A **Task** is a specific assignment given to an agent. It defines what needs to be done, what the result should look like, who does it, and optionally what prior task outputs it depends on.

```
TASK
  description     ->  Detailed instructions for the agent
  expected_output ->  Clear specification of the desired result
  agent           ->  Which agent handles this task

  Optional:
  context         ->  List of tasks whose output feeds this task
  output_pydantic ->  Force structured Pydantic output
  output_file     ->  Write output to a file
  async_execution ->  Run in parallel (non-blocking)
  human_input     ->  Pause and request human feedback
```

## Chaining Tasks with Context

The `context` parameter passes the output of previous tasks to the current task automatically:

```
research_task  ------>  analysis_task  ------>  write_task
    |                        |                       |
    |   (research output)    |   (analysis report)   |
    +------------------------+-----------------------+
                      via context=[...]
```

## Structured Output

Use `output_pydantic` to enforce a typed, machine-readable result:

```python
class Report(BaseModel):
    title: str
    summary: str
    key_findings: List[str]

task = Task(..., output_pydantic=Report)
result = crew.kickoff()
print(result.pydantic.key_findings)  # typed access
```

In [ ]:
from crewai import Task
from pydantic import BaseModel
from typing import List

# Example 1: Basic task
research_task = Task(
    description=(
        "Research the latest developments in Large Language Models (LLMs) "
        "from the past 6 months. Focus on:\n"
        "- New model releases and their capabilities\n"
        "- Key technical breakthroughs\n"
        "- Industry adoption trends\n"
        "Provide at least 5 notable developments with brief explanations."
    ),
    expected_output=(
        "A structured report with 5+ LLM developments. "
        "For each: model/paper name, release date, key capability. "
        "Include a brief summary paragraph at the end."
    ),
    agent=researcher,
)

# Example 2: Task with context — receives output of research_task
analysis_task = Task(
    description=(
        "Using the provided research, analyze the trends and identify "
        "the top 3 most impactful developments. For each, explain:\n"
        "- Why it matters to developers and businesses\n"
        "- Potential use cases\n"
        "- Any limitations or concerns"
    ),
    expected_output="A concise analysis report (400-600 words) in markdown format",
    agent=writer,
    context=[research_task],
)

# Example 3: Structured output with Pydantic
class ArticleOutline(BaseModel):
    title: str
    subtitle: str
    sections: List[str]
    estimated_word_count: int

outline_task = Task(
    description=(
        "Create a detailed article outline about the AI developments "
        "from the research. Target audience: software developers."
    ),
    expected_output="A structured article outline with title, subtitle, sections, and estimated word count",
    agent=writer,
    context=[research_task, analysis_task],
    output_pydantic=ArticleOutline,
)

print("Tasks defined.")
print(f"  research_task  : agent={research_task.agent.role}")
print(f"  analysis_task  : agent={analysis_task.agent.role}, context={[t.description for t in analysis_task.context]}")
print(f"  outline_task   : output_pydantic={outline_task.output_pydantic.__name__}")

# 4: Crews

## What is a Crew?

A **Crew** orchestrates a team of agents executing a set of tasks under a defined process. It manages execution order, passes task outputs as context, and aggregates the final result.

```
CREW
  agents   ->  [researcher, analyst, writer]
  tasks    ->  [research_task, analysis_task, write_task]
  process  ->  Process.sequential | Process.hierarchical

  Optional:
  memory       ->  Enable shared memory across agents
  cache        ->  Cache tool call results
  verbose      ->  Detailed execution logging
  max_rpm      ->  API rate limiting
  embedder     ->  Custom embedding model (used with memory)
  manager_llm  ->  LLM for the hierarchical manager agent
```

## Kickoff Methods

| Method | Description |
|--------|-------------|
| `crew.kickoff()` | Synchronous run, returns result |
| `crew.kickoff(inputs={"topic": "AI"})` | Pass dynamic placeholder values |
| `crew.kickoff_for_each(inputs=[...])` | Run for a list of inputs |
| `await crew.akickoff()` | Native async execution |

## Reading Results

```python
result = crew.kickoff()
print(result.raw)         # plain text output
print(result.pydantic)    # Pydantic model (if output_pydantic was set)
print(result.to_dict())   # dict representation
print(result.token_usage) # token usage statistics
```

In [ ]:
from crewai import Agent, Task, Crew, Process

# Agents
demo_researcher = Agent(
    role="Research Analyst",
    goal="Find key information about the requested topic",
    backstory="Expert researcher with deep knowledge of current AI trends.",
    llm="gemini/gemini-2.5-flash",
    verbose=False,
)

demo_writer = Agent(
    role="Technical Writer",
    goal="Transform research into clear, structured content",
    backstory="Technical writer who specializes in making AI topics accessible.",
    llm="gemini/gemini-2.5-flash",
    verbose=False,
)

# Tasks — use {topic} as a placeholder, filled at kickoff
demo_research_task = Task(
    description="Research 3 key facts about {topic} that would interest developers.",
    expected_output="3 well-explained facts about {topic}, each 2-3 sentences",
    agent=demo_researcher,
)

demo_write_task = Task(
    description="Write a short developer briefing about {topic} using the research.",
    expected_output="A 200-word markdown briefing with a title and bullet points",
    agent=demo_writer,
    context=[demo_research_task],
)

# Crew
demo_crew = Crew(
    agents=[demo_researcher, demo_writer],
    tasks=[demo_research_task, demo_write_task],
    process=Process.sequential,
    verbose=True,
)

print("Crew assembled.")
print(f"  Agents  : {[a.role for a in demo_crew.agents]}")
print(f"  Tasks   : {len(demo_crew.tasks)}")
print(f"  Process : {demo_crew.process}")

# Uncomment to run (requires GOOGLE_API_KEY):
# result = await demo_crew.akickoff(inputs={"topic": "vector databases"})
# print(result.raw)

# 5: Process Types

## Sequential Process (Default)

Tasks execute **one after another in a fixed order**. Each task's processing and output is automatically passed as context to the next task.

```
  Task 1            Task 2            Task 3
(Researcher)  -->  (Analyst)   -->   (Writer)
     |                  |                |
     +------ output ----+---- output ----+
```

Best for: linear pipelines with clear task dependencies.

## Hierarchical Process

A **manager agent** plans, delegates tasks to the right agents, validates outputs, and iterates. Tasks are dynamically allocated — not pre-assigned.

```
                Manager Agent
               /         |      \
         Agent 1      Agent 2      Agent 3
       (Researcher)  (Analyst)    (Writer)
```

Best for: complex projects where task allocation should be dynamic, or where quality validation between steps is needed.

## Comparison

| Feature | Sequential | Hierarchical |
|---------|-----------|--------------|
| Task order | Fixed | Dynamic (manager decides) |
| `manager_llm` required | No | Yes |
| Task pre-assignment | Yes | No (manager allocates) |
| Built-in validation | No | Yes (manager validates) |
| Overhead | Low | Higher (extra LLM calls) |

In [ ]:
! pip install crewai crewai-tools -q


import os
from google.colab import userdata

# Set your Google API key
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
# MODEL = "gemini/gemma-4-31b-it"
MODEL = "gemini/gemini-3.5-flash"

In [ ]:
from crewai import Agent, Task, Crew, Process

# Shared agents
agent_researcher = Agent(
    role="Market Researcher",
    goal="Gather comprehensive data on the requested topic",
    backstory="10 years of market research experience, skilled at synthesizing information.",
    llm=MODEL,
    verbose=False,
)

agent_analyst = Agent(
    role="Data Analyst",
    goal="Identify patterns and insights from research data",
    backstory="Statistician turned business analyst, expert at spotting trends.",
    llm=MODEL,
    verbose=False,
)

agent_writer = Agent(
    role="Business Writer",
    goal="Produce executive-level written reports",
    backstory="MBA graduate with 8 years writing board-level reports.",
    llm=MODEL,
    verbose=False,
)

# Tasks
task1 = Task(
    description="Research the current state of {topic} — key players, market size, growth rate.",
    expected_output="A structured research summary with data points",
    agent=agent_researcher,
)
task2 = Task(
    description="Identify the top 3 opportunities and risks in {topic}.",
    expected_output="3 opportunities and 3 risks, each with rationale",
    agent=agent_analyst,
    context=[task1],
)
task3 = Task(
    description="Write an executive summary of findings for a C-suite audience.",
    expected_output="A 300-word executive summary in professional tone",
    agent=agent_writer,
    context=[task1, task2],
)

# Option A: Sequential
sequential_crew = Crew(
    agents=[agent_researcher, agent_analyst, agent_writer],
    tasks=[task1, task2, task3],
    process=Process.sequential,
    verbose=False,
)

# Option B: Hierarchical — manager dynamically allocates tasks
hierarchical_crew = Crew(
    agents=[agent_researcher, agent_analyst, agent_writer],
    tasks=[task1, task2, task3],
    process=Process.hierarchical,
    manager_llm="gemini/gemini-3.5-flash",
    # https://docs.crewai.com/v1.15.5/en/learn/custom-manager-agent
    verbose=True,
)

print("Sequential crew  :", sequential_crew.process)
print("Hierarchical crew:", hierarchical_crew.process, "| manager_llm:", hierarchical_crew.manager_llm)

result = hierarchical_crew.kickoff_async(inputs={"topic": "AI infrastructure"})

In [ ]:
result = await result

In [ ]:
result

# The Mental Framework for Agentic AI


| Component | Mental Framework                     | Key Features                                              |
|-----------|---------------------------------|-----------------------------------------------------------|
| Crew      | The top-level organization/department      | - Manages AI agent teams<br>- Oversees workflows<br>- Ensures collaboration<br>- Delivers outcomes |
| AI Agents | Specialized team members        | - Have specific roles <br>- Use designated tools<br>- Can delegate tasks<br>- Make autonomous decisions |
| Process   | Workflow management system      | - Defines collaboration patterns<br>- Controls task assignments<br>- Manages interactions<br>- Ensures efficient execution |
| Tasks     | Individual assignments          | - Have clear objectives<br>- Use specific tools<br>- Feed into larger process<br>- Produce actionable results |

- Data Science Product Research Department (Crew)
  - Market Researcher (Agent)
  - Data Analyst (Agent)
  - Data Scientist (Agent)
  - Manager (Orchestrating Agent)
  

  - Tasks
    - Finding new improvements in existing systems/processes available in the market: Market Researcher
    - Collecting data: Market Researcher
        - Internet Search Capability (Tools)
        - Web Scraping Capability (Tools)
        - Storage organization Capability (Tools)

    - Analyzing data: Data Analyst
    - RnD on new tech: Data Scientist
    - Creating reports: Manager
    - Publishing reports & data: Manager



*   HR Department (Crew)
    *   Recruiter (Agent)
    *   Onboarding Specialist (Agent)
    *   People Management (Agent)
    *   Payroll Managers (Agent)
    *   Event Manager (Agent)
    *   L&D Manager (Agent)
    *   Find and Analyze resumes: Recruiter
        *   Internet Search Capability
        *   PDF reading and searching capability
    *   Collaborate and create JDs: Recruiter
    *   Do first-cut and cultural-fit interviews: Recruiter
    *   Schedule interviews: Recruiter
    *   Policy management: Onboarding Specialist





# 6: Tools

## What are Tools?

Tools extend agents with the ability to interact with the external world — searching the web, reading files, calling APIs, executing queries, and more.

```
TOOLS ECOSYSTEM

  Built-in (crewai-tools):
    SerperDevTool        -- Google search
    FileReadTool         -- Read local files
    WebsiteSearchTool    -- RAG search on any website
    PDFSearchTool        -- Search PDF content
    CSVSearchTool        -- Query CSV data
    DirectoryReadTool    -- List/read directory contents
    GithubSearchTool     -- Search GitHub repositories

  Custom:
    @tool decorator      -- Quick single-function tools
    BaseTool subclass    -- Full control, input validation, async support

  MCP Tools:             -- See Section 11
    MCPServerAdapter     -- Connect any MCP-compatible server as tools
```

## Custom Tool Approaches

| Approach | Best For |
|----------|----------|
| `@tool` decorator | Simple, stateless functions |
| `BaseTool` subclass | Input validation, state, async, configuration |

### BaseTool Pattern
```python
class MyToolInput(BaseModel):
    query: str = Field(..., description="Search query")

class MyTool(BaseTool):
    name: str = "My Tool"
    description: str = "What this tool does and when to use it."
    args_schema: Type[BaseModel] = MyToolInput

    def _run(self, query: str) -> str:
        # implementation
        return result
```

In [ ]:
from crewai import Agent
from crewai.tools import tool, BaseTool
from pydantic import BaseModel, Field
from typing import Type

# -------------------------------------------------------------------
# Method 1: @tool decorator — quick, concise
# -------------------------------------------------------------------

@tool("Word Counter")
def count_words(text: str) -> str:
    """Count words, sentences, and characters in the given text."""
    words = len(text.split())
    sentences = text.count('.') + text.count('!') + text.count('?')
    chars = len(text)
    return f"Words: {words}, Sentences: {sentences}, Characters: {chars}"


@tool("Unit Converter")
def convert_units(value: float, from_unit: str, to_unit: str) -> str:
    """Convert between units: km/miles, kg/lbs, celsius/fahrenheit."""
    conversions = {
        ("km", "miles"): lambda x: x * 0.621371,
        ("miles", "km"): lambda x: x * 1.60934,
        ("kg", "lbs"): lambda x: x * 2.20462,
        ("lbs", "kg"): lambda x: x / 2.20462,
        ("celsius", "fahrenheit"): lambda x: x * 9/5 + 32,
        ("fahrenheit", "celsius"): lambda x: (x - 32) * 5/9,
    }
    key = (from_unit.lower(), to_unit.lower())
    if key in conversions:
        result = conversions[key](value)
        return f"{value} {from_unit} = {result:.4f} {to_unit}"
    return f"Conversion from {from_unit} to {to_unit} is not supported."


# -------------------------------------------------------------------
# Method 2: BaseTool subclass — full control + input schema
# -------------------------------------------------------------------

class CalculatorInput(BaseModel):
    """Input schema for the Safe Calculator."""
    expression: str = Field(
        ...,
        description="A mathematical expression, e.g. '(10 + 5) * 2' or '2 ** 8'"
    )


class CalculatorTool(BaseTool):
    name: str = "Safe Calculator"
    description: str = "Evaluates safe mathematical expressions using basic arithmetic."
    args_schema: Type[BaseModel] = CalculatorInput

    def _run(self, expression: str) -> str:
        allowed = set("0123456789+-*/.() ** ")
        if not all(c in allowed for c in expression):
            return "Error: expression contains disallowed characters."
        try:
            result = eval(expression, {"__builtins__": {}})
            return f"{expression} = {result}"
        except Exception as e:
            return f"Error: {e}"


# -------------------------------------------------------------------
# Test tools directly — no agent needed
# -------------------------------------------------------------------

print("--- Word Counter ---")
print(count_words.run("Hello world. This is a test! How many words?"))

print("\n--- Unit Converter ---")
print(convert_units.run(**{"value": 100.0, "from_unit": "km", "to_unit": "miles"}))

print("\n--- Safe Calculator ---")
calc = CalculatorTool()
print(calc._run("(10 + 5) * 2 - 3"))
print(calc._run("2 ** 10"))

# -------------------------------------------------------------------
# Attach tools to an agent
# -------------------------------------------------------------------
agent_with_tools = Agent(
    role="Data Processor",
    goal="Process and analyze text and numerical data accurately",
    backstory="A meticulous data professional who always double-checks calculations.",
    llm="gemini/gemini-2.5-flash",
    tools=[count_words, convert_units, calc],
    verbose=True,
)

print(f"\nAgent '{agent_with_tools.role}' has {len(agent_with_tools.tools)} tools: "
      f"{[t.name for t in agent_with_tools.tools]}")

# 7: Memory

## What is Memory in CrewAI?

Memory gives agents the ability to **remember facts across tasks and across separate crew runs**. The unified `Memory` class uses a local vector store (LanceDB) and an LLM to automatically organise, store, and retrieve information.

```
MEMORY STORE (LanceDB, local)
  /
    /project
      /project/decisions
      /project/findings
    /agent
      /agent/researcher
      /agent/writer

Recall scoring:
  composite = (semantic_weight   x similarity)
            + (recency_weight    x decay)
            + (importance_weight x importance)
```

## Three Usage Modes

| Mode | Code | Use When |
|------|------|----------|
| Standalone | `memory = Memory()` | Scripts, notebooks |
| Crew-shared | `Crew(..., memory=True)` | All agents share one pool |
| Agent-scoped | `memory.scope("/agent/x")` | Agent needs private context |

## Memory vs. Task Context

| | Memory | Task Context |
|---|--------|--------------|
| Persists across runs | Yes | No |
| Retrieval | Semantic and Relevance search | Direct output pass |
| Scope | Global / hierarchical | Per-task chain |

In [ ]:
from crewai import Memory
help(Memory)

In [ ]:
from crewai import Memory, Agent, Task, Crew, Process

# -------------------------------------------------------------------
# Configure Memory with Google embeddings
# -------------------------------------------------------------------

memory = Memory(
    llm="gemini/gemini-2.5-flash",
    embedder={
        "provider": "google-generativeai",
        "config": {"model_name": "gemini-embedding-001"}
    },
    recency_weight=0.5,
    semantic_weight=0.3,
    importance_weight=0.2,
    recency_half_life_days=7,
)

print("Memory configuration:")
print(f"  LLM      : gemini/gemini-2.5-flash")
print(f"  Embedder : gemini-embedding-001")
print(f"  Weights  : recency={memory.recency_weight}, "
      f"semantic={memory.semantic_weight}, "
      f"importance={memory.importance_weight}")

# Local/offline alternative (no API key needed):
# memory = Memory(
#     llm="ollama/llama3.2",
#     embedder={"provider": "ollama", "config": {"model_name": "mxbai-embed-large"}}
# )

# -------------------------------------------------------------------
# Crew with shared memory — agents share one memory pool
# -------------------------------------------------------------------

mem_researcher = Agent(
    role="Knowledge Researcher",
    goal="Research topics and store key findings",
    backstory="Expert researcher who builds institutional knowledge.",
    llm="gemini/gemini-2.5-flash",
    verbose=False,
)

mem_writer = Agent(
    role="Report Writer",
    goal="Write reports using accumulated knowledge",
    backstory="Writer who leverages past research to produce rich reports.",
    llm="gemini/gemini-2.5-flash",
    verbose=False,
)

task1 = Task(
    description="Research 3 key facts about {topic}.",
    expected_output="3 key facts, clearly stated",
    agent=mem_researcher,
)
task2 = Task(
    description="Write a brief report on {topic} drawing on any known background.",
    expected_output="A 150-word report",
    agent=mem_writer,
)

memory_crew = Crew(
    agents=[mem_researcher, mem_writer],
    tasks=[task1, task2],
    process=Process.sequential,
    memory=True,
    verbose=False,
)

print("\nMemory-enabled crew ready.")
print(f"  memory={memory_crew.memory}")

# Run twice to observe memory accumulation across runs:
# result1 = await memory_crew.kickoff_aysnc(inputs={"topic": "transformer architecture"})
# print(f"  memory={memory_crew.memory}")
# result2 = await memory_crew.kickoff_aysnc(inputs={"topic": "attention mechanisms"})
# print(f"  memory={memory_crew.memory}")

# -------------------------------------------------------------------
# Agent-scoped memory — agent sees only its own subtree
# -------------------------------------------------------------------

memory2 = Memory(
    llm="gemini/gemini-2.5-flash",
    embedder={"provider": "google-generativeai", "config": {"model_name": "gemini-embedding-001"}}
)

scoped_agent = Agent(
    role="Private Researcher",
    goal="Maintain confidential research notes",
    backstory="A researcher who keeps findings isolated from other agents.",
    llm="gemini/gemini-2.5-flash",
    memory=memory2.scope("/agent/private_researcher"),
    verbose=False,
)

print("\nScoped agent memory configured at: /agent/private_researcher")

# 8: Knowledge Sources

## What is Knowledge?

Knowledge sources let you inject **static domain content** — documents, PDFs, policies, product catalogs, FAQs — directly into agents or crews. Agents retrieve relevant chunks at query time using RAG (Retrieval-Augmented Generation).

```
KNOWLEDGE PIPELINE

  Source (string / file / PDF / CSV)
      |
      v  chunked + embedded
  Vector Store
      |
      v  semantic search at task time
  Agent Context (injected into task prompt)
```

## Available Source Types

| Class | Input | Use Case |
|-------|-------|----------|
| `StringKnowledgeSource` | Raw string | Policies, FAQs, quick facts |
| `TextFileKnowledgeSource` | `.txt` paths | Internal docs, notes |
| `PDFKnowledgeSource` | `.pdf` paths | Reports, manuals, papers |
| `CSVKnowledgeSource` | `.csv` paths | Data tables, catalogs |
| `JSONKnowledgeSource` | `.json` paths | Structured records |

## Scope

- **Crew-level** `knowledge_sources=[...]` — all agents in the crew can access it  
- **Agent-level** `knowledge_sources=[...]` — only that specific agent can access it

In [ ]:
! pip install -q crewai crewai-tools

In [ ]:
import os
from google.colab import userdata

# Set your Google API key
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

In [ ]:
from crewai import Agent, Task, Crew, Process
from crewai.knowledge.source.string_knowledge_source import StringKnowledgeSource

# -------------------------------------------------------------------
# Define knowledge sources
# -------------------------------------------------------------------

company_policy = """
ACME Corp AI Usage Policy - Version 2.0 (2025)

1. APPROVED MODELS
   - Google Gemini 2.5 Flash is approved for internal use.
   - Claude 3.5 Sonnet is approved for customer-facing applications.
   - All local models must be reviewed by the Security team first.

2. DATA HANDLING
   - Never send PII to external LLM APIs.
   - All prompts containing customer data must use the on-premises deployment.
   - Conversation logs must be encrypted and purged after 90 days.

3. RATE LIMITS
   - Development: max 100 API calls per hour.
   - Production: max 10,000 API calls per hour.
   - Exceeding limits requires CTO approval.

4. COST MANAGEMENT
   - Monthly AI spend limit per team: $500.
   - Projects exceeding $2,000/month require VP Engineering sign-off.

5. SECURITY
   - API keys must be stored in the company secrets vault.
   - Never hardcode API keys in source code.
   - Rotate API keys every 90 days.
"""

product_catalog = """
ACME Corp Product Catalog - AI Tools Division

PRODUCT: CrewManager Pro
  Price: $299/month per workspace
  Features: Unlimited agents, 100GB memory, priority support
  Best for: Enterprises with complex multi-agent workflows

PRODUCT: CrewManager Starter
  Price: $49/month per workspace
  Features: Up to 5 agents, 5GB memory, email support
  Best for: Small teams and startups

PRODUCT: CrewManager Open Source
  Price: Free (self-hosted)
  Features: Core features, community support
  Best for: Developers, researchers, personal projects

PRODUCT: AI Consulting Package
  Price: From $5,000/engagement
  Features: Architecture design, implementation, team training
  Best for: Companies new to multi-agent systems
"""

policy_source = StringKnowledgeSource(content=company_policy)
catalog_source = StringKnowledgeSource(content=product_catalog)

print("Knowledge sources created.")
print(f"  Policy : {len(company_policy)} chars")
print(f"  Catalog: {len(product_catalog)} chars")

# -------------------------------------------------------------------
# Agent with knowledge — answers questions using the documents
# -------------------------------------------------------------------

support_agent = Agent(
    role="ACME Corp Support Specialist",
    goal="Answer customer questions accurately using official company documentation",
    backstory=(
        "You are an ACME Corp support specialist. You provide accurate answers "
        "based on official policies and product documentation only. "
        "You never guess or make up information."
    ),
    llm="gemini/gemini-2.5-flash",
    knowledge_sources=[policy_source, catalog_source],  # agent-level
    verbose=True,
    # knowledge=[policy_source] # agent-level
)


sales_agent = Agent(
    role="Sales Representative",
    goal="Help customers choose the right product",
    backstory="Experienced sales rep who knows the product line inside out.",
    llm="gemini/gemini-2.5-flash",
    verbose=False,
)


task1 = Task(
    description="Answer this customer question: '{question}'",
    expected_output="A clear, accurate answer grounded in company documentation",
    agent=support_agent,
)

# -------------------------------------------------------------------
# Crew-level knowledge — shared across all agents in the crew
# -------------------------------------------------------------------

knowledge_crew = Crew(
    agents=[support_agent, sales_agent],
    tasks=[task1],
    knowledge_sources=[policy_source, catalog_source],  # crew-level
    process=Process.sequential,
    verbose=False,
    embedder={
        "provider": "google-generativeai",
        "config": {"model_name": "gemini-embedding-001"}
    }
)

print(f"\nKnowledge-enabled crew: {len(knowledge_crew.knowledge_sources)} source(s)")

# Run:
# result = knowledge_crew.kickoff(inputs={"question": "What is the monthly spend limit per team?"})
# print(result.raw)

# 9: Project 1 — AI Newsletter Generator

## Objective

Build a production-ready **weekly AI newsletter generator** using a sequential crew of four specialised agents.

## Architecture

```
INPUT: topic, edition date
      |
      v
  Task 1 — News Researcher
  Finds 5 significant recent AI developments

      |
      v
  Task 2 — Trend Analyst
  Selects top 3 stories, writes hooks, rates importance

      |
      v
  Task 3 — Newsletter Writer
  Writes full newsletter edition (500 words)

      |
      v
  Task 4 — Copy Editor
  Polishes tone, fixes issues, enforces length

      |
      v
OUTPUT: publication-ready newsletter (markdown)
```

## Agents

| Agent | Role | Responsibility |
|-------|------|---------------|
| News Researcher | Finds raw stories | Monitors papers, blogs, launches |
| Trend Analyst | Curates and ranks | Picks what matters most to practitioners |
| Newsletter Writer | Drafts content | Structures and writes the edition |
| Copy Editor | Quality control | Polishes grammar, tone, and length |

In [ ]:
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool
from datetime import datetime
from crewai_tools import SerperDevTool

# -------------------------------------------------------------------
# Tools
# -------------------------------------------------------------------

import os
from google.colab import userdata

# Set your Google API key
os.environ["SERPER_API_KEY"] = userdata.get('SERPER_API_KEY')
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

@tool("Current Date")
def get_current_date() -> str:
    """Returns today's date in DD Month, YYYY format."""
    return datetime.now().strftime("%d %B, %Y")


# -------------------------------------------------------------------
# Agents
# -------------------------------------------------------------------

news_researcher = Agent(
    role="AI News Researcher",
    goal="Find the 5 most significant AI developments published in the last 30 days",
    backstory=(
        "You are a dedicated AI industry researcher who monitors research papers, "
        "company blogs, product launches, and news outlets daily. "
        "Your expertise is separating genuine breakthroughs from marketing noise."
    ),
    llm="gemini/gemini-3.5-flash",
    tools=[get_current_date, SerperDevTool()],
    verbose=True,
)

trend_analyst = Agent(
    role="AI Trend Analyst",
    goal="Select and rank the most impactful stories for a technical audience",
    backstory=(
        "Former ML engineer turned industry analyst. You understand both the technical "
        "details and business implications of AI developments. "
        "You curate what practitioners and decision-makers care about most."
    ),
    llm="gemini/gemini-3.5-flash",
    verbose=True,
)

newsletter_writer = Agent(
    role="AI Newsletter Writer",
    goal="Write an engaging, informative weekly newsletter edition",
    backstory=(
        "You have written the 'AI Weekly Digest' for 3 years, growing it from 500 to "
        "50,000 subscribers. Your style is conversational yet substantive — you explain "
        "complex topics without oversimplifying, and always include practical takeaways."
    ),
    llm="gemini/gemini-3.5-flash",
    verbose=True,
)

copy_editor = Agent(
    role="Senior Copy Editor",
    goal="Polish the newsletter to publication quality",
    backstory=(
        "20 years editing technical publications. Sharp eye for clarity, flow, and accuracy. "
        "You catch factual errors, fix awkward phrasing, and keep the newsletter's voice consistent."
    ),
    llm="gemini/gemini-3.5-flash",
    verbose=True,
)


# -------------------------------------------------------------------
# Tasks
# -------------------------------------------------------------------

research_task = Task(
    description=(
        "Research the latest developments in {topic} as of {date}.\n\n"
        "Find and summarise:\n"
        "1. Recent model releases or major updates (last 30 days)\n"
        "2. Breakthrough research papers or technical developments\n"
        "3. Major industry partnerships or acquisitions\n"
        "4. Regulatory or policy developments\n"
        "5. Open-source releases or tooling advancements\n\n"
        "For each item include: what happened, why it matters, key players involved."
    ),
    expected_output=(
        "5 well-researched news items. "
        "Each item: 3-5 sentences covering the event, significance, and key actors."
    ),
    agent=news_researcher,
    human_input=True
)

selection_task = Task(
    description=(
        "From the research findings, select the TOP 3 stories most relevant to "
        "AI practitioners and developers. For each selected story:\n"
        "- Explain why it was selected (impact / novelty / relevance)\n"
        "- Write a 2-sentence hook that makes readers want to learn more\n"
        "- Rate importance: HIGH or MEDIUM"
    ),
    expected_output="3 selected stories, each with a hook and importance rating",
    agent=trend_analyst,
    context=[research_task],
)

write_task = Task(
    description=(
        "Write the {date} edition of 'The AI Weekly' based on the curated stories.\n\n"
        "Required structure:\n"
        "1. THIS WEEK IN AI — 3-sentence intro capturing the week's theme\n"
        "2. TOP STORIES — 3 stories, each with: headline, 100-word summary, 'Why it matters'\n"
        "3. QUICK HITS — 3 additional noteworthy items as bullet points\n"
        "4. THOUGHT OF THE WEEK — one provocative question or insight\n"
        "5. SIGN OFF — brief closing line\n\n"
        "Tone: informative, slightly opinionated, accessible to ML engineers and technical PMs. "
        "Target length: ~500 words."
    ),
    expected_output="A complete newsletter edition (~500 words) in clean html",
    agent=newsletter_writer,
    context=[selection_task],
)

edit_task = Task(
    description=(
        "Review and polish the newsletter draft:\n"
        "- Fix grammar and style issues\n"
        "- Ensure consistent tone throughout\n"
        "- Make headlines more specific and compelling\n"
        "- Verify all claims are grounded in the provided research\n"
        "- Confirm total length is 450-550 words"
    ),
    expected_output="The final, publication-ready newsletter in html",
    agent=copy_editor,
    context=[write_task],
)


# -------------------------------------------------------------------
# Crew
# -------------------------------------------------------------------

newsletter_crew = Crew(
    agents=[news_researcher, trend_analyst, newsletter_writer, copy_editor],
    tasks=[research_task, selection_task, write_task, edit_task],
    process=Process.sequential,
    verbose=True,
)

print("Newsletter crew assembled.")
print(f"  Agents : {[a.role for a in newsletter_crew.agents]}")
print(f"  Tasks  : {len(newsletter_crew.tasks)}")
print(f"  Process: {newsletter_crew.process}")

result = await newsletter_crew.kickoff_async(inputs={
    "topic": "Artificial Intelligence",
    "date": get_current_date.run()
})
print(result.raw)

In [ ]:
print(result.raw)

---

# 10: Project 2 — Background Verification System

## Objective

Build a **Background Verification (BGV) system** that processes a candidate's submitted documents and produces a structured verification report.

## Architecture

```
INPUT: candidate profile (name, education, employment, ID details)
      |
      v
  Task 1 — Document Analyst
  Extracts and normalises all candidate data from the submission

      |
      v
  Task 2 — Education Verifier
  Verifies degree, institution, graduation year

      |
      v
  Task 3 — Employment Verifier
  Verifies job titles, companies, tenure, gaps

      |
      v
  Task 4 — Identity Verifier
  Validates government ID details, checks consistency

      |
      v
  Task 5 — BGV Report Compiler
  Produces structured BGVReport (Pydantic output)

      |
      v
OUTPUT: BGVReport { status, issues, recommendation }
```

## Agents

| Agent | Responsibility |
|-------|---------------|
| Document Analyst | Extract and structure all submitted information |
| Education Verifier | Cross-check academic credentials |
| Employment Verifier | Validate employment history and detect gaps |
| Identity Verifier | Validate identity document details |
| BGV Report Compiler | Synthesise findings into a structured report |

In [ ]:
from crewai import Agent, Task, Crew, Process
from pydantic import BaseModel
from typing import List

# -------------------------------------------------------------------
# Output schema — structured BGV report
# -------------------------------------------------------------------

class BGVReport(BaseModel):
    candidate_name: str
    education_status: str       # "Verified" | "Discrepancy Found" | "Unable to Verify"
    education_notes: str
    employment_status: str      # "Verified" | "Gap Detected" | "Discrepancy Found"
    employment_notes: str
    identity_status: str        # "Verified" | "Mismatch Found" | "Incomplete"
    identity_notes: str
    issues: List[str]           # list of specific issues found, empty if none
    overall_status: str         # "Clear" | "Flagged" | "Rejected"
    recommendation: str


# -------------------------------------------------------------------
# Agents
# -------------------------------------------------------------------

document_analyst = Agent(
    role="Document Analyst",
    goal="Extract and structure all candidate information from the submitted profile",
    backstory=(
        "You are a meticulous document processing specialist with 8 years of experience "
        "in HR and compliance. You read candidate submissions and organise all information "
        "into clearly structured categories: personal details, education, employment, identity."
    ),
    llm="gemini/gemini-2.5-flash",
    verbose=True,
)

education_verifier = Agent(
    role="Education Verification Specialist",
    goal="Verify the accuracy of educational qualifications claimed by the candidate",
    backstory=(
        "Former university registrar turned background screening specialist. "
        "You know what legitimate degree certificates look like, how to spot "
        "inconsistencies in institution names, graduation years, and course details. "
        "You flag discrepancies without jumping to conclusions."
    ),
    llm="gemini/gemini-2.5-flash",
    verbose=True,
)

employment_verifier = Agent(
    role="Employment History Verifier",
    goal="Validate the candidate's employment history, detect gaps, and flag discrepancies",
    backstory=(
        "Experienced HR investigator specialising in employment verification. "
        "You cross-check job titles, company names, tenure dates, and reporting structures. "
        "You are skilled at identifying unexplained employment gaps and inflated titles."
    ),
    llm="gemini/gemini-2.5-flash",
    verbose=True,
)

identity_verifier = Agent(
    role="Identity Verification Specialist",
    goal="Validate government-issued identity documents and ensure consistency across the submission",
    backstory=(
        "Former KYC (Know Your Customer) analyst with expertise in identity document validation. "
        "You verify that names, dates of birth, and ID numbers are consistent across all "
        "submitted documents and match the candidate profile."
    ),
    llm="gemini/gemini-2.5-flash",
    verbose=True,
)

report_compiler = Agent(
    role="BGV Report Compiler",
    goal="Synthesise all verification findings into a clear, structured BGV report",
    backstory=(
        "Senior background verification manager with 12 years of experience. "
        "You read verification reports from multiple specialists and produce an authoritative "
        "summary report with a clear overall status and actionable recommendation."
    ),
    llm="gemini/gemini-2.5-flash",
    verbose=True,
)


# -------------------------------------------------------------------
# Tasks
# -------------------------------------------------------------------

extract_task = Task(
    description=(
        "Review the following candidate submission and extract all information "
        "in a structured format:\n\n"
        "{candidate_submission}\n\n"
        "Extract:\n"
        "- Full name and personal details\n"
        "- Educational qualifications (institution, degree, year, grade)\n"
        "- Employment history (company, role, duration, reason for leaving)\n"
        "- Identity documents provided (type, number, issuing authority)"
    ),
    expected_output=(
        "A structured summary of all candidate information under the headings: "
        "Personal Details, Education, Employment History, Identity Documents."
    ),
    agent=document_analyst,
)

education_task = Task(
    description=(
        "Using the extracted candidate information, verify the education details:\n"
        "- Are institution names real and correctly spelled?\n"
        "- Are the graduation years plausible given the candidate's age?\n"
        "- Are there any inconsistencies between the degree name and institution?\n"
        "- Note any details that cannot be independently verified from the submission alone.\n\n"
        "Do NOT fabricate verification results. Base your assessment only on the provided data."
    ),
    expected_output=(
        "Education verification findings: status (Verified / Discrepancy Found / "
        "Unable to Verify), specific observations, and any issues noted."
    ),
    agent=education_verifier,
    context=[extract_task],
)

employment_task = Task(
    description=(
        "Using the extracted candidate information, verify the employment history:\n"
        "- Are there unexplained gaps of more than 3 months?\n"
        "- Are job titles consistent with the stated companies and industries?\n"
        "- Does the chronological order of roles make sense?\n"
        "- Are there any overlapping employment dates?\n\n"
        "Do NOT fabricate verification results. Base your assessment only on the provided data."
    ),
    expected_output=(
        "Employment verification findings: status (Verified / Gap Detected / "
        "Discrepancy Found), specific observations, and any issues noted."
    ),
    agent=employment_verifier,
    context=[extract_task],
)

identity_task = Task(
    description=(
        "Using the extracted candidate information, verify the identity documents:\n"
        "- Is the name on ID documents consistent with the name in the application?\n"
        "- Is the ID number format valid for the stated document type and country?\n"
        "- Are there any inconsistencies between documents?\n"
        "- Flag any missing or incomplete identity information.\n\n"
        "Do NOT fabricate verification results. Base your assessment only on the provided data."
    ),
    expected_output=(
        "Identity verification findings: status (Verified / Mismatch Found / Incomplete), "
        "specific observations, and any issues noted."
    ),
    agent=identity_verifier,
    context=[extract_task],
)

report_task = Task(
    description=(
        "Compile all verification findings into a final BGV report.\n\n"
        "Overall status rules:\n"
        "- 'Clear': all three checks are Verified with no issues\n"
        "- 'Flagged': at least one check has a minor issue requiring follow-up\n"
        "- 'Rejected': any check has a Discrepancy Found or critical issue\n\n"
        "Recommendation must be one of: "
        "'Proceed with onboarding', 'Proceed with caution — follow up required', "
        "'Do not proceed — escalate to HR'."
    ),
    expected_output="A complete BGVReport with all required fields filled in accurately",
    agent=report_compiler,
    context=[education_task, employment_task, identity_task],
    output_pydantic=BGVReport,
)


# -------------------------------------------------------------------
# Crew
# -------------------------------------------------------------------

bgv_crew = Crew(
    agents=[document_analyst, education_verifier, employment_verifier,
            identity_verifier, report_compiler],
    tasks=[extract_task, education_task, employment_task, identity_task, report_task],
    process=Process.sequential,
    verbose=True,
)

print("BGV crew assembled.")
print(f"  Agents : {[a.role for a in bgv_crew.agents]}")
print(f"  Tasks  : {len(bgv_crew.tasks)}")

# -------------------------------------------------------------------
# Sample candidate submission
# -------------------------------------------------------------------

sample_candidate = """
CANDIDATE SUBMISSION

Name: Priya Sharma
Date of Birth: 14 March 1995

EDUCATION
  Degree: Bachelor of Technology, Computer Science
  Institution: Indian Institute of Technology, Bombay
  Year of Graduation: 2017
  Grade: 8.4 / 10

EMPLOYMENT HISTORY
  Role: Software Engineer
  Company: Infosys Limited
  Duration: July 2017 - December 2019

  Role: Senior Software Engineer
  Company: Flipkart Internet Pvt. Ltd.
  Duration: January 2020 - August 2022

  Role: Staff Engineer
  Company: Razorpay Software Pvt. Ltd.
  Duration: September 2022 - Present

IDENTITY DOCUMENTS
  Type: Passport
  Number: P1234567
  Issuing Country: India
  Expiry: March 2030
"""

print("\nSample candidate submission ready.")

# Run the BGV crew:
# result = bgv_crew.kickoff(inputs={"candidate_submission": sample_candidate})
#
# Access structured output:
# report = result.pydantic
# print(f"Candidate     : {report.candidate_name}")
# print(f"Overall Status: {report.overall_status}")
# print(f"Recommendation: {report.recommendation}")
# if report.issues:
#     print("Issues found  :")
#     for issue in report.issues:
#         print(f"  - {issue}")

# 11: MCP Tools Integration

## What is MCP?

**Model Context Protocol (MCP)** is an open standard that allows AI agents to connect to external tools and data sources through a unified interface. Any service that exposes an MCP-compatible server can instantly become a tool for your CrewAI agents — no custom integration code required.

## How it Works in CrewAI

```
AGENT
  tools=[...mcp_tools...]
       |
       v
  MCPServerAdapter  (crewai-tools)
       |
       +---- Streamable HTTP  http://host/mcp
       +---- SSE              http://host/sse
       +---- Stdio            local process (npx, python, etc.)
```

The `MCPServerAdapter` from `crewai_tools` connects to one or more MCP servers and exposes all their tools as standard CrewAI tools. It manages the connection lifecycle automatically via a context manager.

## Transport Types

| Transport | Config Key | Use Case |
|-----------|-----------|----------|
| Streamable HTTP | `{"url": "...", "transport": "streamable-http"}` | Remote production servers |
| SSE | `{"url": "...", "transport": "sse"}` | Remote streaming servers |
| Stdio | `StdioServerParameters(command=..., args=[...])` | Local processes (npm, Python) |

## Popular Public MCP Servers

| Server | Install | Provides |
|--------|---------|----------|
| Filesystem | `@modelcontextprotocol/server-filesystem` | Read/write local files |
| Fetch | `@modelcontextprotocol/server-fetch` | HTTP requests |
| Brave Search | `@modelcontextprotocol/server-brave-search` | Web search |
| GitHub | `@modelcontextprotocol/server-github` | Repo read/write |
| PostgreSQL | `@modelcontextprotocol/server-postgres` | DB queries |

In [ ]:
from crewai import Agent, Task, Crew, Process
from crewai_tools import MCPServerAdapter
from mcp import StdioServerParameters
import os

# -------------------------------------------------------------------
# Example 1: Single HTTP MCP server
# -------------------------------------------------------------------
# Connects to a running Streamable HTTP MCP server.
# The 'with' block handles connection setup and teardown automatically.

def run_with_http_mcp_server():
    server_config = {
        "url": "http://localhost:8000/mcp",
        "transport": "streamable-http"
    }

    with MCPServerAdapter(server_config) as mcp_tools:
        print(f"Tools from HTTP server: {[t.name for t in mcp_tools]}")

        analyst = Agent(
            role="Data Analyst",
            goal="Analyse data using available external tools",
            backstory="Expert analyst with access to live data sources.",
            llm="gemini/gemini-2.5-flash",
            tools=mcp_tools,
            verbose=True,
        )

        task = Task(
            description="Use the available tools to answer: {question}",
            expected_output="A clear, data-grounded answer",
            agent=analyst,
        )

        crew = Crew(agents=[analyst], tasks=[task], process=Process.sequential)
        return crew.kickoff(inputs={"question": "What is the current system status?"})


# -------------------------------------------------------------------
# Example 2: Stdio MCP server — local filesystem access
# -------------------------------------------------------------------
# Connects to the official MCP filesystem server via npx.
# Requires: npm and @modelcontextprotocol/server-filesystem installed: npm i @modelcontextprotocol/server-filesystem


def run_with_filesystem_mcp():
    server_params = StdioServerParameters(
        command="npx",
        args=["-y", "@modelcontextprotocol/server-filesystem", "/tmp/workspace"],
        env={**os.environ},
    )

    with MCPServerAdapter(server_params) as mcp_tools:
        print(f"Filesystem MCP tools: {[t.name for t in mcp_tools]}")

        file_agent = Agent(
            role="File Manager",
            goal="Read and write files in the workspace directory",
            backstory="DevOps engineer with expertise in file system operations.",
            llm="gemini/gemini-2.5-flash",
            tools=mcp_tools,
            verbose=True,
        )

        task = Task(
            description="List all files in the workspace and summarise their contents.",
            expected_output="A summary of files found and their key contents",
            agent=file_agent,
        )

        crew = Crew(agents=[file_agent], tasks=[task], process=Process.sequential)
        return crew.kickoff()


# -------------------------------------------------------------------
# Example 3: Multiple MCP servers aggregated
# -------------------------------------------------------------------
# Combines tools from multiple servers into a single agent.

def run_with_multiple_mcp_servers():
    server_configs = [
        {"url": "http://localhost:8001/mcp", "transport": "streamable-http"},
        {"url": "http://localhost:8002/sse", "transport": "sse"},
        StdioServerParameters(
            command="npx",
            args=["-y", "@modelcontextprotocol/server-fetch"],
            env={**os.environ},
        ),
    ]

    with MCPServerAdapter(server_configs) as aggregated_tools:
        print(f"Aggregated tools ({len(aggregated_tools)}): "
              f"{[t.name for t in aggregated_tools]}")

        versatile_agent = Agent(
            role="Research Assistant",
            goal="Use all available tools to answer complex questions",
            backstory="Versatile analyst with access to web, data, and file systems.",
            llm="gemini/gemini-2.5-flash",
            tools=aggregated_tools,
            verbose=True,
        )
        # Build crew and run as needed...


# -------------------------------------------------------------------
# Note on usage
# -------------------------------------------------------------------
# These functions require running MCP servers. To test locally:
#
#   npm install -g @modelcontextprotocol/server-filesystem
#   npx @modelcontextprotocol/server-filesystem /tmp/workspace
#
# Then call: run_with_filesystem_mcp()
#
# The MCPServerAdapter pattern is identical regardless of server type —
# swap the config dict and the tools are automatically discovered.

print("MCP integration examples defined.")
print("MCPServerAdapter automatically discovers and wraps all tools from connected servers.")
print("Use as a context manager to ensure clean connection lifecycle management.")

# 12: Skills in CrewAI

## What are Skills?

**Skills** are instruction packs that give coding agents (Claude Code, Cursor, Codex, Copilot) deep, up-to-date knowledge about CrewAI conventions — how to scaffold crews, configure agents, use tools, and follow framework best practices.

Skills live on [skills.sh](https://skills.sh/crewaiinc/skills) and are installed in one command. They reduce back-and-forth with your coding agent by injecting the right context upfront.

## Why Use Skills?

Without skills, you must explain CrewAI patterns every session. With skills installed, your coding agent already knows:
- How to create agents with the right structure
- How to configure tasks and context chains
- How to use `@CrewBase` and YAML configuration
- Current best practices for memory, knowledge, and tools
- How to avoid common mistakes

## How to Install

```bash
# Install via npx (works with Claude Code, Cursor, Codex, Copilot)
npx skills add crewaiinc/skills

# Or install via the Claude Code plugin marketplace
# Search for "crewaiinc/skills"
```

## Skill Coverage

The official `crewaiinc/skills` pack covers:

| Area | What the agent learns |
|------|-----------------------|
| **Agents** | Role/goal/backstory patterns, LLM selection, delegation |
| **Tasks** | Context chaining, structured output, human input |
| **Crews** | Process types, memory, caching, rate limiting |
| **Tools** | `@tool` decorator, `BaseTool`, MCP integration |
| **Knowledge** | Source types, crew vs agent scope |
| **Project Layout** | `agents/`, `tasks/`, `tools/` directory conventions |
| **CLI** | `crewai create`, `crewai run`, `crewai reset-memories` |

## Relationship to Knowledge Sources

Skills and Knowledge sources both provide context — but they serve different purposes:

| | Skills | Knowledge Sources |
|---|--------|-------------------|
| **Purpose** | Teach coding agents how to write CrewAI code | Give runtime agents domain-specific facts |
| **Used by** | Your IDE / coding assistant | Your CrewAI agents at execution time |
| **Content** | Framework patterns and conventions | Business documents, policies, data |
| **Install** | `npx skills add` | `StringKnowledgeSource(...)` etc. |

## Using Skills Programmatically

Skills can also be attached to agents as `knowledge_sources` at runtime, providing CrewAI domain knowledge to agents that generate or reason about CrewAI configurations:

```python
from crewai import Agent
from crewai.knowledge.source.string_knowledge_source import StringKnowledgeSource

# Embed a skill as a knowledge source for a meta-agent
crewai_skill_content = """
CrewAI Agent Pattern:
  agent = Agent(
      role="...",         # job title / domain expertise
      goal="...",         # what it aims to produce
      backstory="...",    # qualifications and working style
      llm="...",          # model string e.g. gemini/gemini-2.5-flash
      tools=[...],        # list of Tool instances
      verbose=True,
  )
...
"""

skill_source = StringKnowledgeSource(content=crewai_skill_content)

scaffold_agent = Agent(
    role="CrewAI Architect",
    goal="Design and generate correct CrewAI crew configurations",
    backstory="Expert in CrewAI framework conventions and best practices.",
    llm="gemini/gemini-2.5-flash",
    knowledge_sources=[skill_source],
    verbose=True,
)
```